In [ ]:
!pip install pyproj
!pip install -U numpy==1.26.4
!pip install transformers
!pip install -U accelerate
!pip install pandas
!pip install PyPDF2
!pip install NLTK
!pip install editdistance
!pip install paramiko
!pip install func_timeout

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [1]:
# import libraries
import urllib.request
import requests
import json
from pyproj import Transformer
import os
import time
import sys
from datetime import datetime
from xml.etree import ElementTree as ET
import logging
import nltk
import pandas as pd
import re

# import common functions
sys.path.insert(1, '../')
import common

# set name of module, to fetch info from config
module_name = "archis"





/home/alex/anaconda3/envs/agnes-bert-file-import/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False
cuda gpu number is 0


Some weights of the model checkpoint at alexbrandsen/ArcheoBERTje-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/home/alex/anaconda3/envs/agnes-bert-file-import/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more 

In [6]:
# get info from config file
config = common.get_config()

# # set up logging
# log_location = config['data_source'][module_name]['harvest_log_location']
# now = datetime.now()
# date = now.strftime("%Y-%m-%d")
# logfile = f"{log_location}harvest-log-{module_name}-{date}.log"
# logging.basicConfig(level=logging.DEBUG, filename=logfile, filemode="a+",
#                 format="%(asctime)-15s %(levelname)-8s %(message)s")

# log config info        
#pdf_folder = config['data_source'][module_name]['pdf_folder']
pdf_folder = '/media/alex/Data/agnes_data/archis/Archis_rapporten_2025/docs'
print(f'pdf_folder: {pdf_folder}')

json_folder = config['data_source'][module_name]['json_folder']
print(f'json_folder: {json_folder}')

html_folder = config['data_source'][module_name]['html_folder']
print(f'html_folder: {html_folder}')

language = config['data_source'][module_name]['language']
print(f'language: {language}')

bert_model = config['bert_models'][language]
print(f'bert_model: {bert_model}')


pdf_folder: /media/alex/Data/agnes_data/archis/Archis_rapporten_2025/docs
json_folder: /media/alex/Data/agnes_data/archis/json/
html_folder: /media/alex/Data/agnes_data/archis/html/
language: dutch
bert_model: /media/alex/Data/agnes_models/ArcheoBERTje-NER


In [5]:
archis_docinfo_location = '/media/alex/Data/agnes_data/archis/Archis_rapporten_2025/archis3_documentinformatie_xml.csv'

archis_docinfo = pd.read_csv(archis_docinfo_location, sep=";", encoding="latin-1")
print(archis_docinfo)

                                                bestand  document_id  \
0     Z2323285_30129769-afm-1709638079355-300866 Rap...          NaN   
1     Z2323285_30129769-afm-1709639667124-300866 Rap...          NaN   
2     Z2338895_30129769-afm-1716996379950-Briefrappo...          NaN   
3     Z2344929_40408504-afm-1694770763199-2011 AWN J...          NaN   
4     Z2395595_30129769-afm-1698330842459-GAR 1319 S...          NaN   
...                                                 ...          ...   
7815  Z5684965_40408504-afm-1738270188176-20250130_2...          NaN   
7816  Z5684965_40408504-afm-1738270188176-20250130_2...          NaN   
7817  Z5684965_40408504-afm-1738270188176-20250130_2...          NaN   
7818  Z5688083_40408504-afm-1739134270904-20250209_2...          NaN   
7819  Z5688091_40408504-afm-1739135099097-20250209_2...          NaN   

                auteur                                              titel  \
0      F.M.J. Delporte  IJzertijd en Romeinse bewoning op 

In [10]:


for directory, subdirectories, files in os.walk(pdf_folder):
    for file in files:
        
        file_location = os.path.join(directory, file)

        # if not done yet
        # file_name = common.cleanFileName(file)
        # archis_zaakidentificatie = int(file.split('_')[0][1:]+'100')
        # doc_id = f"{archis_zaakidentificatie}_{file_name.replace('.pdf','')}"
        # json_output_folder = f"{json_folder}/{doc_id}"
        # if os.path.exists(json_output_folder):
        #     #print('Output folder for '+file+' already exists, skipping')
        #     continue #skip this file if it already exists
            
        print(f"indexing: {file}")
        

        archis_zaakidentificatie = int(file.split('_')[0][1:]+'100')
        #print(archis_zaakidentificatie)
        
        docinfo = archis_docinfo.loc[archis_docinfo['bestand'] == file]
        
        if len(docinfo) == 0: # no result in db, log and skip
            print(f"no entry in db for {archis_zaakidentificatie}, skipping")
            continue
            
        #if len(docinfo) > 1: # multiple rows with same filename, get first one (already done above with .iloc[0])
        #    docinfo = docinfo.loc[docinfo['document_id'].notnull()]

        #print(docinfo['titel'])
        
        file_name = common.cleanFileName(file)

        output_document = {}
        output_document['source'] = 'archis'
        output_document['file_name'] = file_name
        output_document['file_type'] = 'report'
        output_document['title'] = docinfo['titel'].values[0]
        if pd.notna(docinfo['auteur'].values[0]): # check if auteur is empty
            creators = re.split(',|&| en |/|;', docinfo['auteur'].values[0])    # split on muliple characters, because messy data         
            output_document['creators'] = creators
        output_document['description'] = '' # no descriptions in this data source
        output_document['publisher'] = docinfo['uitvoerder'].values[0]
        if pd.notna(docinfo['jaar'].values[0]): # check if jaar is empty
            output_document['createdAt'] = int(docinfo['jaar'].values[0])
        output_document['identifiers'] = {
            'uri': docinfo['link'].values[0],
            'archis_zaakidentificatie': int(docinfo['zaakidentificatie'].values[0]),
            #'archis_zaak_id': int(zaakdocument['zaak_id'].values[0]),
            #'archis_identificatie': str(zaakdocument['identificatie'].values[0])
        }
        output_document['language'] = 'Dutch'
        output_document['html_folder_name'] = f"{archis_zaakidentificatie}_{file_name.replace('.pdf','')}"

        #coordinates
        if str(docinfo['x_coordinaat'].values[0]) != 'nan' and str(docinfo['x_coordinaat'].values[0]) != 'nan':
            coordX = int(docinfo['x_coordinaat'].values[0])
            coordY = int(docinfo['y_coordinaat'].values[0])
            lat, lon = common.rd2wgs(coordX,coordY)
            output_document['coordX'] = coordX
            output_document['coordY'] = coordY
            output_document['location'] = {'lat':lat,'lon':lon}

        # set document identifier
        doc_id = f"{archis_zaakidentificatie}_{file_name.replace('.pdf','')}"
        print(f"doc_id: {doc_id}")

        # save document.json 
        json_output_folder = f"{json_folder}/{doc_id}"
        common.savejson(output_document, f"{json_output_folder}/document.json")

        print(f"saved doc json")

        # process pdf, store page.json files with entities 
        common.run_ner_on_pdf(
            file_location, 
            json_output_folder, 
            bert_model, 
            language
        )

        print(f"ran NER, saved page json")

        # process pdf, save html files
        html_output_folder = f"{html_folder}/{doc_id}"
        common.pdf2html(file_location, html_output_folder)

        print(f"generated and saved html")



print('done!')


Xref table not zero-indexed. ID numbers for objects will be corrected.


indexing: Z5624262_08177178-afm-1736507620382-2024-0406_BO_Farmsum_Oosterhorn (noor.pdf
7261    Archeologisch bureauonderzoek (BO), Farmsum, O...
Name: titel, dtype: object
doc_id: 5624262100_Z5624262_08177178-afm-1736507620382-2024-0406_BO_Farmsum_Oosterhorn_noor
saved doc json


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


ran NER, saved page json
pdftohtml error for file /media/alex/Data/agnes_data/archis/Archis_rapporten_2025/docs/Z5624262_08177178-afm-1736507620382-2024-0406_BO_Farmsum_Oosterhorn (noor.pdf /bin/sh: 1: Syntax error: "(" unexpected

generated and saved html
indexing: Z5501100_30129769-afm-1731683543062-NL24-648800269-103268.pdf
6161    Archeologisch onderzoek N321 te Gassel en Esch...
Name: titel, dtype: object
doc_id: 5501100100_Z5501100_30129769-afm-1731683543062-NL24-648800269-103268
saved doc json
ran NER, saved page json
Converted /media/alex/Data/agnes_data/archis/Archis_rapporten_2025/docs/Z5501100_30129769-afm-1731683543062-NL24-648800269-103268.pdf to html
generated and saved html
indexing: Z5314997_32098920-afm-1723542944895-Rap 5779_000956_Wageningen Herinricht.pdf
3416    Herinrichting Benedenbuurt, Wageningen, gemeen...
Name: titel, dtype: object
doc_id: 5314997100_Z5314997_32098920-afm-1723542944895-Rap_5779_000956_Wageningen_Herinricht
saved doc json
ran NER, saved page j

Xref table not zero-indexed. ID numbers for objects will be corrected.


...

Converted /media/alex/Data/agnes_data/archis/Archis_rapporten_2025/docs/Z5183177_14117581-afm-1712151880099-ArcheoPro rapport Hedelstaete Hedel 2.pdf to html
generated and saved html
indexing: Z5151813_12063933-afm-1704458542844-AM21521_Droogsestraat (ong.pdf
1434    Archeologisch bureau- en verkennend veldonderz...
Name: titel, dtype: object
doc_id: 5151813100_Z5151813_12063933-afm-1704458542844-AM21521_Droogsestraat_ong
saved doc json
ran NER, saved page json
pdftohtml error for file /media/alex/Data/agnes_data/archis/Archis_rapporten_2025/docs/Z5151813_12063933-afm-1704458542844-AM21521_Droogsestraat (ong.pdf /bin/sh: 1: Syntax error: "(" unexpected

generated and saved html
indexing: Z5260734_60810688-afm-1720611211165-22020094 Rapportage BO IVO Sint Oeden.pdf
2168    Sint-Oedenrode, Kerkstraat 11 Gemeente Meierij...
Name: titel, dtype: object
doc_id: 5260734100_Z5260734_60810688-afm-1720611211165-22020094_Rapportage_BO_IVO_Sint_Oeden
saved doc json
ran NER, saved page json
Convert

Xref table not zero-indexed. ID numbers for objects will be corrected.


Converted /media/alex/Data/agnes_data/archis/Archis_rapporten_2025/docs/Z5488961_08080701-afm-1710505936033-V-23.pdf to html
generated and saved html
indexing: Z5134009_32098920-afm-1709642355737-Rap 5642_4230781_Krimpenerwaard Ouder.pdf
1137    Dorpsstraat 106, Ouderkerk aan den IJssel (gem...
Name: titel, dtype: object
doc_id: 5134009100_Z5134009_32098920-afm-1709642355737-Rap_5642_4230781_Krimpenerwaard_Ouder
saved doc json
ran NER, saved page json
Converted /media/alex/Data/agnes_data/archis/Archis_rapporten_2025/docs/Z5134009_32098920-afm-1709642355737-Rap 5642_4230781_Krimpenerwaard Ouder.pdf to html
generated and saved html
indexing: Z5122223_34137810-afm-1696863039529-RAAPrap_6036_TIKON4_20221014.pdf
979    Plangebied Koning Willem II College, gemeente ...
Name: titel, dtype: object
doc_id: 5122223100_Z5122223_34137810-afm-1696863039529-RAAPrap_6036_TIKON4_20221014
saved doc json
ran NER, saved page json
Converted /media/alex/Data/agnes_data/archis/Archis_rapporten_2025/docs/Z5

Object 0 0 not defined.
Object 0 0 not defined.
Object 0 0 not defined.
Object 0 0 not defined.
Object 0 0 not defined.
Object 0 0 not defined.
Object 0 0 not defined.
Object 0 0 not defined.
Object 0 0 not defined.
Object 0 0 not defined.
Object 0 0 not defined.
Object 0 0 not defined.
Object 0 0 not defined.
Object 0 0 not defined.


Converted /media/alex/Data/agnes_data/archis/Archis_rapporten_2025/docs/Z5507014_13038286-afm-1734427902041-22612_002 rapport archeologisch proef.pdf to html
generated and saved html
indexing: Z5631317_28071689-afm-1726128313718-Bijlage_II_Profielkolommen.pdf
7399    Een houtskoolmeiler in Udenhout. Een proefsleu...
Name: titel, dtype: object
doc_id: 5631317100_Z5631317_28071689-afm-1726128313718-Bijlage_II_Profielkolommen
saved doc json
Could not find object.
Could not find object.
Could not find object.
Could not find object.
Could not find object.
Could not find object.
Could not find object.
Could not find object.
Could not find object.
Could not find object.
Could not find object.
Could not find object.
Could not find object.
Could not find object.
ran NER, saved page json
Converted /media/alex/Data/agnes_data/archis/Archis_rapporten_2025/docs/Z5631317_28071689-afm-1726128313718-Bijlage_II_Profielkolommen.pdf to html
generated and saved html
indexing: Z5450599_75235153-afm-1705498

In [3]:
# upload json and html to webserver
common.upload2webserver(json_folder, html_folder, module_name, config['webserver']['json_folder'], config['webserver']['html_folder'])

print(f"uploaded json/html to webserver")

# remotely start indexing script on webserver
common.start_index(module_name)

print(f"indexing on webserver started")

print(f"done!")

uploaded json/html to webserver
indexing on webserver started
done!
